### Import code from a file
In order to make workflow components that are easy to maintain, reliable and reusable, the workflow components can only use code that is in the notebook cell and declared dependencies.
In order to use your own models in NaaVRE workflows, you can create python or R packages. Python and R packages can be installed into virtual labs. Adding your package to conda forge ensures it is simple to install your package in NaaVRE.
Under some circumstances it will be hard to turn your model into a Python or R package. In that case the NaaVRE development team can onboard your model in a different way.  
During development it may be desirable to reuse classes and functions in multiple workflow components without turning them into packages.
In that case you can load code from a file.

To load code from files, do the following:
- Put the code you want to use in multiple workflow components in a `.py` or `.r` files.
- Save the `.py` or `.r` file in [cloud storage](https://naavre.net/docs/NaaVRE_documentation/read-write-files/#Cloud-storage).
  - If you don't know where to put the file in cloud storage, store it in your personal cloud storage(`naa-vre-user-data`).
  - If you want to share the files with other users, you need to request a virtual lab bucket or write access to the public bucket.
- Match `param_cloud_storage_folder` to the folder where you've saved the files. 
- Match `param_first_file_to_import` with the name of the file you've saved in cloud storage.
  - You can use more than one file. In that case, also use `param_second_file_to_import` and optionally add more parameters.
  - Ensure that if objects in the second file depend on objects in the first file, the module from the first file is loaded before the module from the second file. In this example, we call a function from `timedelta_utilities.py` that uses a function from `datetime_utilities.py`.
- Replace `# Custom code` with an import and usage of an object from a file you specified.
- Run the code in this notebook to check if it works.

In [ ]:
# parameters and configurations
param_datetime = "2026-12-31 23:59:59"

# Import parameters
param_cloud_storage_folder = 'naa-vre-public/training-materials' # could also be set to 'naa-vre-user-data'
param_first_file_to_import = 'datetime_utilities.py'
param_second_file_to_import = 'timedelta_utilities.py'

conf_cloud_storage = '/home/jovyan/Cloud Storage'

In [ ]:
# Workflow component using modules from files

# Boilerplate code to import modules from files
#########################################################################################
import importlib
from importlib import util as importlib_util
import sys
import pathlib

def import_module_from_file(file_folder: str, filename: str):
    filepath = pathlib.Path(file_folder) / filename
    module_name = filename.rsplit('.', 1)[0]
    spec = importlib_util.spec_from_file_location(module_name, filepath)
    module = importlib_util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    print(f"imported {module_name} from {filepath}")
    return importlib.import_module(module_name)

cloud_storage_path = pathlib.Path(conf_cloud_storage) / param_cloud_storage_folder
#########################################################################################

# Custom code
datetime_utilities = import_module_from_file(cloud_storage_path, param_first_file_to_import)
timedelta_utilities = import_module_from_file(cloud_storage_path, param_second_file_to_import)

timedelta_utilities.print_time_difference(param_datetime)